# 4D. Keras-only CBOW Recommender
Train CBOW embeddings (as in 4B), build a document vector per tweet by averaging
word embeddings, then recommend most-similar tweets to a query via cosine similarity.

In [1]:
import numpy as np
import pandas as pd
import re
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
df = pd.read_csv("../../datasets/CBOW/twitter_dataset.csv")
texts = df["Text"].dropna().tolist()

sentences = []
for tweet in texts:
    tokens = re.findall(r"[a-z]+", str(tweet).lower())
    if len(tokens) >= 5:
        sentences.append(tokens)

print("tweets:", len(sentences))

tweets: 10000


In [3]:
vocab = {}
for sent in sentences:
    for word in sent:
        if word not in vocab:
            vocab[word] = len(vocab)
vocab_size = len(vocab)

In [4]:
window = 2
contexts = []
targets = []
for sent in sentences:
    ids = [vocab[w] for w in sent]
    for i in range(window, len(ids) - window):
        context = ids[i-window:i] + ids[i+1:i+window+1]
        contexts.append(context)
        targets.append(ids[i])

contexts = np.array(contexts)
targets = np.array(targets)

In [5]:
model = keras.Sequential([
    layers.Input(shape=(2*window,)),
    layers.Embedding(vocab_size, 50, name="embed"),
    layers.GlobalAveragePooling1D(),
    layers.Dense(vocab_size, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
history = model.fit(contexts, targets, epochs=3, batch_size=256)

Epoch 1/3


   1/1172 ━━━━━━━━━━━━━━━━━━━━ 5:54 303ms/step - loss: 6.8779

  37/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8784    

  73/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8783

 110/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8784

 147/1172 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 6.8784

... (101 outputs left out)

1095/1172 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 6.8513

1129/1172 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 6.8512

1163/1172 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 6.8512

1172/1172 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 6.8511


In [6]:
embeddings = model.get_layer("embed").get_weights()[0]

# build a mean-embedding doc vector per tweet
doc_vecs = []
for sent in sentences:
    ids = [vocab[w] for w in sent]
    vecs = embeddings[ids]
    doc_vecs.append(vecs.mean(axis=0))
doc_vecs = np.array(doc_vecs)

# normalise for cosine
doc_norms = np.linalg.norm(doc_vecs, axis=1, keepdims=True)
doc_vecs_norm = doc_vecs / (doc_norms + 1e-9)
print("doc matrix:", doc_vecs.shape)

doc matrix: (10000, 50)


In [7]:
query = "happy day great"
q_tokens = re.findall(r"[a-z]+", query.lower())
q_ids = [vocab[w] for w in q_tokens if w in vocab]
q_vec = embeddings[q_ids].mean(axis=0)
q_vec = q_vec / (np.linalg.norm(q_vec) + 1e-9)

sims = doc_vecs_norm @ q_vec
top = sims.argsort()[::-1][:5]

print(f"query: {query!r}\n")
# find the original tweet text for each top index
original_texts = [t for t in texts if len(re.findall(r"[a-z]+", str(t).lower())) >= 5]
for i in top:
    print(f"  [{sims[i]:.3f}] {original_texts[i][:120]}")

query: 'happy day great'



  [0.507] Race central body. Determine enter tax difference.
Explain role blue boy lot strong. Staff girl nature fear beautiful lo
  [0.492] Often specific mention. Example describe break.
Middle drive finish. Daughter blue bill bit though try toward. Consider 
  [0.484] Later score hard customer. Interesting executive happy budget project nice art specific.
No state respond. Respond withi
  [0.477] Just specific color. Hour federal open act. Particular visit wide another family sea.
Art security win great. School dis
  [0.477] Usually particular entire cut. Less shake worker. Happy yourself up ask the keep.
Ago visit affect everyone eye be. Cond
